In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm

In [2]:


# --------------------------
# 0) Load data
# --------------------------
PANEL_PATH = "data/processed/merton_panel.csv"
OVX_PATH = 'data/processed/Macroeconomic_variables/OVXCLS.csv'
OIL_PATH = 'data/processed/OIL/oil_prices_datastream.csv'

df = pd.read_csv(PANEL_PATH)
ovx = pd.read_csv(OVX_PATH)
oil = pd.read_csv(OIL_PATH)

df["date"] = pd.to_datetime(df["date"])
ovx["Date"] = pd.to_datetime(ovx["Date"])
oil["Date"] = pd.to_datetime(oil["Date"])

ovx = ovx.merge(oil, on='Date')


ovx = ovx.sort_values("Date").rename(columns={"Date": "date", "OVXCLS": "ovx"})
ovx["ovx"] = pd.to_numeric(ovx["ovx"], errors="coerce")
ovx = ovx.dropna(subset=["ovx"])

df = df.sort_values(["country_clean","date"]).reset_index(drop=True)

# Exporter flag (adjust if needed)
df["exporter"] = (df["group"].astype(str).str.contains("export", case=False, na=False)).astype(int)

ovx


/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_98602/1035816076.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  oil["Date"] = pd.to_datetime(oil["Date"])


,date,ovx,Brent,WTI,OPEC_basket,Dubai_Crude
0,2007-05-10,27.09,65.62,61.82,62.36,63.10
1,2007-05-11,26.41,66.24,62.38,63.15,64.04
2,2007-05-14,27.23,66.79,62.47,63.79,64.08
3,2007-05-15,27.89,67.35,63.18,63.79,65.08
4,2007-05-16,27.07,67.91,62.56,64.55,64.22
...,...,...,...,...,...,...
4598,2024-12-24,30.35,73.75,70.87,73.13,74.59
4600,2024-12-26,30.01,73.75,70.38,73.92,74.31
4601,2024-12-27,30.21,74.03,71.28,73.91,75.25
4602,2024-12-30,30.77,74.38,71.73,74.62,75.48


In [3]:

# --------------------------
# 1) Align daily OVX to weekly dates (last obs on/before date)
# --------------------------
weekly_dates = df[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)
weekly_ovx = pd.merge_asof(weekly_dates, ovx, on="date", direction="backward")
weekly_ovx["ovx"] = weekly_ovx["ovx"].ffill()

df = df.merge(weekly_ovx, on="date", how="left")
df["ovx"] = df["ovx"].ffill()


In [4]:
df['Brent'] = pd.to_numeric(df['Brent'], errors="coerce")
oil_weekly = df[["date", 'Brent']].drop_duplicates().sort_values("date").reset_index(drop=True)
oil_weekly["log_oil"] = np.log(oil_weekly['Brent'])
oil_weekly["r_oil"] = oil_weekly["log_oil"].diff()

# Define downside jump weeks as bottom q quantile of weekly oil returns
OIL_DOWN_Q = 0.05
cut = oil_weekly["r_oil"].dropna().quantile(OIL_DOWN_Q)
oil_weekly["Jminus"] = (oil_weekly["r_oil"] <= cut).astype(int)

df = df.merge(oil_weekly[["date", "r_oil", "Jminus"]], on="date", how="left")
df["Jminus"] = df["Jminus"].fillna(0).astype(int)
print(f"Oil downside jump definition: Jminus_t = 1{{ weekly oil log-return <= {cut:.4f} }} (bottom {OIL_DOWN_Q:.2%})")
print("Share of weeks flagged as downside jumps:", df[["date","Jminus"]].drop_duplicates()["Jminus"].mean())



Oil downside jump definition: Jminus_t = 1{ weekly oil log-return <= -0.0820 } (bottom 5.00%)
Share of weeks flagged as downside jumps: 0.050522648083623695


In [6]:
# --------------------------
# 4) Core variables for structural model
# --------------------------
EPS = 1e-12
H = 5.0  # years horizon requested

df["A"] = pd.to_numeric(df["msci_index"], errors="coerce").clip(lower=EPS)
df["B"] = pd.to_numeric(df["default_barrier"], errors="coerce").clip(lower=EPS)
df["sigma_annual"] = pd.to_numeric(df["msci_vol_52w"], errors="coerce").clip(lower=1e-6)
df["mu_annual"] = pd.to_numeric(df["rf"], errors="coerce").fillna(0.0)

df["cds"] = pd.to_numeric(df["cds_spread"], errors="coerce")
df["log_cds"] = np.log(df["cds"].clip(lower=EPS))

df = df.dropna(subset=["A","B","sigma_annual","mu_annual","log_cds","country_clean","exporter","Jminus"]).copy()
df = df.sort_values(["country_clean","date"]).reset_index(drop=True)

In [7]:
# --------------------------
# 5) Fix jump size distribution using "jump-like" MSCI return weeks (pooled)
# --------------------------
df["logA"] = np.log(df["A"])
df["r_week"] = df.groupby("country_clean")["logA"].diff()

r = df["r_week"].dropna()
r_std = r.std()
k = 3.0
jump_like = r.abs() > k * r_std
jump_obs = df.loc[jump_like, "r_week"].dropna()

if len(jump_obs) >= 30:
    muJ = float(jump_obs.mean())
    sigJ = float(jump_obs.std(ddof=1))
else:
    # conservative fallback
    muJ = -0.02
    sigJ = 0.08

print(f"Fixed jump size distribution from MSCI returns: muJ={muJ:.4f}, sigJ={sigJ:.4f}, n_jump_obs={len(jump_obs)}")


AssertionError: 

In [ ]:

# --------------------------
# 6) 5-year diffusion-only DD (baseline, closed-form)
# --------------------------
H = 5.0  # years

def dd_diffusion(A, B, mu, sigma, H):
    return (np.log(A/B) + (mu - 0.5*sigma**2)*H) / (sigma*np.sqrt(H))

df["dd_diff_5y"] = dd_diffusion(df["A"], df["B"], df["mu_annual"], df["sigma_annual"], H)


In [ ]:

# --------------------------
# 7) Jump-diffusion PD/DDD via fast terminal-horizon Monte Carlo
#    log A_T = log A_0 + (mu - 0.5 sigma^2)H + sigma sqrt(H) Z + Sum_{k=1}^N J_k
#    where N ~ Poisson(lambda*H) and J_k ~ Normal(muJ, sigJ)
#
#    Jump intensity depends on tail regime at t and exporter status:
#    lambda_it = lambda0 * exp(eta0*tail_t + eta1*exporter_i*tail_t)
# --------------------------
rng = np.random.default_rng(123)

def pd_jump_mc(A0, B, mu, sigma, H, lam, n_paths=4000):
    """
    Monte Carlo PD over horizon H for one observation.
    Returns PD.
    """
    # Draw N for each path
    N = rng.poisson(lam * H, size=n_paths)

    # Draw jump sums efficiently:
    # Sum of N normals ~ Normal(N*muJ, sqrt(N)*sigJ)
    # Handle N=0 -> sum=0
    jump_sum = np.zeros(n_paths, dtype=float)
    idx = N > 0
    if np.any(idx):
        jump_sum[idx] = rng.normal(loc=N[idx]*muJ, scale=np.sqrt(N[idx])*sigJ)

    Z = rng.normal(size=n_paths)
    logAT = np.log(A0) + (mu - 0.5*sigma**2)*H + sigma*np.sqrt(H)*Z + jump_sum
    AT = np.exp(logAT)

    return float(np.mean(AT < B))

def compute_dd_jump_for_params(lambda0, eta0, eta1, n_paths=4000):
    """
    Compute DD^{(J)} for all obs in df given parameters.
    """
    lam = lambda0 * np.exp(eta0*df["tail"].values + eta1*(df["exporter"].values*df["tail"].values))

    PD = np.empty(len(df), dtype=float)

    # Loop is OK for N*T ~ a few thousand; MC is vectorized within each obs.
    # If you have huge T, reduce n_paths or compute only on a subset first.
    A0 = df["A"].values
    B  = df["B"].values
    mu = df["mu_annual"].values
    sg = df["sigma_annual"].values

    for j in range(len(df)):
        PD[j] = pd_jump_mc(A0[j], B[j], mu[j], sg[j], H, lam[j], n_paths=n_paths)

    # Convert PD -> DD (higher is safer); clip PD away from 0/1
    PD = np.clip(PD, 1e-6, 1 - 1e-6)
    DD = -norm.ppf(PD)
    return DD


In [ ]:

# --------------------------
# 8) Country-FE regression helper (robust, numeric-only)
# --------------------------
def fe_regression(y: pd.Series, x: np.ndarray, xname="x"):
    y = pd.to_numeric(y, errors="coerce")
    x = pd.to_numeric(pd.Series(x, index=y.index, name=xname), errors="coerce")

    fe = pd.get_dummies(df["country_clean"].astype("string"), drop_first=True, prefix="cty").astype(float)
    X = pd.concat([x, fe], axis=1)
    X = sm.add_constant(X)

    data = pd.concat([y.rename("y"), X], axis=1).dropna()
    y_clean = data["y"].astype(float).values
    X_clean = data.drop(columns=["y"]).astype(float).values

    model = sm.OLS(y_clean, X_clean).fit()
    resid = model.resid
    sse = float(np.sum(resid**2))
    tss = float(np.sum((y_clean - y_clean.mean())**2))
    r2 = 1.0 - sse / tss if tss > 0 else np.nan

    # fitted values aligned to df index
    yhat = pd.Series(model.fittedvalues, index=data.index)
    return model, sse, r2, yhat

def group_sse(y, yhat, mask):
    resid = (y - yhat).loc[mask].dropna()
    return float(np.sum(resid**2))

mask_exp  = (df["exporter"] == 1)
mask_nexp = (df["exporter"] == 0)

# --------------------------
# 9) Baseline fit: diffusion DD -> log CDS
# --------------------------
m0, sse0, r20, yhat0 = fe_regression(df["log_cds"], df["dd_diff_5y"].values, xname="dd")
sse0_exp  = group_sse(df["log_cds"], yhat0, mask_exp)
sse0_nexp = group_sse(df["log_cds"], yhat0, mask_nexp)

print("\n=== Baseline diffusion DD(5y) ===")
print(f"R2:  {r20:.4f}")
print(f"SSE: {sse0:.4f}")
print(f"Exporters SSE:     {sse0_exp:.4f}")
print(f"Non-exporters SSE: {sse0_nexp:.4f}")


In [ ]:

# --------------------------
# 10) Grid search over jump-intensity parameters (keep it tight at first)
#     lambda0 is jumps per year in normal weeks.
# --------------------------
# Start small for speed; widen once you see where it lands.
lambda0_grid = np.linspace(0.0, 2.0, 9)   # 0 to 2 jumps/year
eta0_grid    = np.linspace(0.0, 1.0, 6)   # common tail multiplier
eta1_grid    = np.linspace(0.0, 1.5, 7)   # extra exporter tail multiplier

N_PATHS = 3000  # increase to 8000-20000 for final numbers once tuned

best = {"lambda0": None, "eta0": None, "eta1": None, "sse": np.inf, "r2": None, "model": None, "yhat": None, "ddj": None}

print("\nGrid search starting... (this can take a bit depending on n_paths and sample size)")

for lam0 in lambda0_grid:
    for e0 in eta0_grid:
        for e1 in eta1_grid:
            ddj = compute_dd_jump_for_params(lam0, e0, e1, n_paths=N_PATHS)
            m, sse, r2, yhat = fe_regression(df["log_cds"], ddj, xname="ddj")
            if sse < best["sse"]:
                best.update(lambda0=lam0, eta0=e0, eta1=e1, sse=sse, r2=r2, model=m, yhat=yhat, ddj=ddj)
                print(f"  New best: SSE={sse:.3f}, R2={r2:.4f} at lambda0={lam0:.2f}, eta0={e0:.2f}, eta1={e1:.2f}")

# Group SSE for best jump model
sse1_exp  = group_sse(df["log_cds"], best["yhat"], mask_exp)
sse1_nexp = group_sse(df["log_cds"], best["yhat"], mask_nexp)

print("\n=== Best jump-intensity model (MC PD -> DD -> log CDS) ===")
print(f"lambda0_hat: {best['lambda0']:.3f} (jumps/year in normal weeks)")
print(f"eta0_hat:    {best['eta0']:.3f} (common tail multiplier)")
print(f"eta1_hat:    {best['eta1']:.3f} (extra exporter tail multiplier)")
print(f"R2:  {best['r2']:.4f}")
print(f"SSE: {best['sse']:.4f}")
print(f"ΔSSE vs baseline: {sse0 - best['sse']:.4f}")

print("\n=== SSE by group ===")
print(f"Exporters:     SSE0={sse0_exp:.4f}  SSE1={sse1_exp:.4f}  Δ={sse0_exp - sse1_exp:.4f}")
print(f"Non-exporters: SSE0={sse0_nexp:.4f}  SSE1={sse1_nexp:.4f}  Δ={sse0_nexp - sse1_nexp:.4f}")

print("\n=== Best jump model mapping coefficients (x1 is DD^J) ===")
print(best["model"].summary().tables[1])


In [ ]:

# --------------------------
# 11) Quick diagnostics: does DD^J worsen (fall) for exporters in tail weeks?
# --------------------------
df["dd_jump_5y_best"] = best["ddj"]
print("\nDD^J summary by exporter x tail:")
print(df.groupby(["exporter","tail"])["dd_jump_5y_best"].describe()[["mean","std","min","max"]])

# --------------------------
# Notes:
# - Increase N_PATHS for stable parameter selection once you see plausible regions.
# - If best params are at the edge of grids, widen grids.
# - Try Q=0.95 and Q=0.99; exporter effect should strengthen if your reduced-form tail story is correct.
# - This is in-sample fit; for "better informs" you should do rolling out-of-sample RMSE by group next.
# --------------------------
